# Great Expectations checks for the Olist pipeline

This notebook develops and runs the **Great Expectations (GX)** checks for the Olist data. Run the cells from top to bottom.

The checks are separated into two groups:

- **Critical checks** - cover invalid business values and impossible sequences, which can block a pipeline run.
- **Observation checks** - monitor volume, completeness, known source anomalies, and distribution changes, to be reviewed without blocking the pipeline immediately.

Primary keys, model-grain duplicates, foreign keys, and transformation logic will be covered by **dbt tests** once the staging models are ready.

## 1. Create a GX Data context

The file context stores reusable suites and validation definitions in the root `gx/` directory. The environment variables are optional and are mainly useful for local verification:

- `GX_SOURCE_MODE=csv` reads the ignored local files in `data/` instead of BigQuery for local verification without having to connect to GCP
- `GX_CONTEXT_MODE=ephemeral` tests the notebook without updating files under `gx/`.

In [53]:
import os
from pathlib import Path

import great_expectations as gx
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

GX_CONTEXT_MODE = os.getenv("GX_CONTEXT_MODE", "file")
if GX_CONTEXT_MODE == "file":
    context = gx.get_context(mode="file", project_root_dir=str(PROJECT_ROOT))
else:
    context = gx.get_context(mode="ephemeral")

print(f"Great Expectations: {gx.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"GX context mode: {GX_CONTEXT_MODE}")

Great Expectations: 1.22.0
Project root: /home/bomberblue/olist-data-pipeline
GX context mode: file


## 2. Choose the data layer

Use `raw` now. When the dbt staging models exist, change `DATA_LAYER` to `staging` and confirm the table names in `TABLE_NAMES["staging"]`.

`PROJECT_ID` identifies the Google Cloud project. `DATASETS` chooses the BigQuery dataset. The notebook does not rely on dbt's `raw_dataset` variable.

In [ ]:
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "olist-data-pipeline-507001")
DATA_LAYER = os.getenv("GX_DATA_LAYER", "raw")
SOURCE_MODE = os.getenv("GX_SOURCE_MODE", "bigquery")

DATASETS = {
    "raw": os.getenv("GX_RAW_DATASET", "olist_raw"),
    "staging": os.getenv("GX_STAGING_DATASET", "olist_staging"),
}

TABLE_NAMES = {
    "raw": {
        "customers": "raw_customers",
        "orders": "raw_orders",
        "order_items": "raw_order_items",
        "payments": "raw_order_payments_dataset",
        "reviews": "raw_order_reviews_dataset",
        "products": "raw_products",
        "sellers": "raw_sellers",
        "geolocation": "raw_geolocation_dataset",
    },
    "staging": {
        "customers": "stg_customers",
        "orders": "stg_orders",
        "order_items": "stg_order_items",
        "payments": "stg_order_payments",
        "reviews": "stg_order_reviews",
        "products": "stg_products",
        "sellers": "stg_sellers",
        "geolocation": "stg_geolocation",
    },
}

CSV_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
}

if DATA_LAYER not in TABLE_NAMES:
    raise ValueError(f"DATA_LAYER must be one of {list(TABLE_NAMES)}")

# local csv can only be used for raw data validation
if SOURCE_MODE == "csv" and DATA_LAYER != "raw":
    raise ValueError("The local CSV option represents the raw layer only.")

print(f"Source: {SOURCE_MODE}")
print(f"Layer: {DATA_LAYER}")
print(f"BigQuery target: {PROJECT_ID}.{DATASETS[DATA_LAYER]}")

Source: bigquery
Layer: raw
BigQuery target: olist-data-pipeline-507001.olist_raw


## 3. Load the eight operational tables

Each DataFrame exists only in notebook memory. A future Dagster asset will repeat this load during every run before invoking the same validations.

In [63]:
def load_frames():
    frames = {}
    if SOURCE_MODE == "bigquery":
        engine = create_engine(f"bigquery://{PROJECT_ID}/{DATASETS[DATA_LAYER]}")
        print(f"engine: {engine}")
        for logical_name, table_name in TABLE_NAMES[DATA_LAYER].items():
            query = f"SELECT * FROM `{PROJECT_ID}.{DATASETS[DATA_LAYER]}.{table_name}`"
            frames[logical_name] = pd.read_sql(query, engine)
    elif SOURCE_MODE == "csv":
        for logical_name, filename in CSV_FILES.items():
            frames[logical_name] = pd.read_csv(PROJECT_ROOT / "data" / filename)
    else:
        raise ValueError("SOURCE_MODE must be 'bigquery' or 'csv'.")
    return frames


frames = load_frames()
display(
    pd.DataFrame(
        [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in frames.items()]
    ).sort_values("table").reset_index(drop=True)
)

engine: Engine(bigquery://olist-data-pipeline-507001/olist_raw)


,table,rows,columns
0,customers,99441,12
1,geolocation,1000163,12
2,order_items,112650,14
3,orders,99441,15
4,payments,103886,12
5,products,32951,16
6,reviews,99224,14
7,sellers,3095,11


## 4. Normalize validation dtypes

BigQuery can return Arrow-backed pandas dtypes. GX 1.22 may reject comparisons when the column and expectation bounds use different numeric types. This cell converts the validation columns to stable pandas numeric, datetime, and string dtypes before creating batches.

In [ ]:
NUMERIC_COLUMNS = {
    "order_items": ["order_item_id", "price", "freight_value"],
    "payments": ["payment_sequential", "payment_installments", "payment_value"],
    "reviews": ["review_score"],
    "products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    ],
    "geolocation": ["geolocation_lat", "geolocation_lng"],
}

DATETIME_COLUMNS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

ZIP_COLUMNS = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}

# rows that had a value before coercion but became null after (i.e. failed to parse)
coercion_failures = []


def record_coercion_failures(table_name, column, original, converted):
    new_nulls = int((original.notna() & converted.isna()).sum())
    if new_nulls:
        coercion_failures.append({"table": table_name, "column": column, "new_nulls": new_nulls})


for table_name, columns in NUMERIC_COLUMNS.items():
    for column in columns:
        original = frames[table_name][column]
        converted = pd.to_numeric(original, errors="coerce").astype("float64")
        record_coercion_failures(table_name, column, original, converted)
        frames[table_name][column] = converted

for table_name, columns in DATETIME_COLUMNS.items():
    for column in columns:
        original = frames[table_name][column]
        converted = pd.to_datetime(original, errors="coerce")
        record_coercion_failures(table_name, column, original, converted)
        frames[table_name][column] = converted

for table_name, column in ZIP_COLUMNS.items():
    original = frames[table_name][column]
    numeric_zip = pd.to_numeric(original, errors="coerce").astype("Int64")
    record_coercion_failures(table_name, column, original, numeric_zip)
    frames[table_name][column] = numeric_zip.astype("string").str.zfill(5)

if coercion_failures:
    print("Coercion introduced new nulls:")
    display(pd.DataFrame(coercion_failures))
else:
    print("No coercion failures: every non-null source value parsed successfully.")

# creates a temporary Boolean column that checks whether the order status agrees with the customer delivery date
# i.e. there should be delivery date if it's already delivered and vice versa
orders = frames["orders"]
orders["delivery_date_matches_status"] = ~(
    ((orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna())
    | ((orders["order_status"] != "delivered") & orders["order_delivered_customer_date"].notna())
)

normalized_dtypes = []
for table_name, columns in NUMERIC_COLUMNS.items():
    normalized_dtypes.extend(
        {"table": table_name, "column": column, "dtype": str(frames[table_name][column].dtype)}
        for column in columns
    )
for table_name, columns in DATETIME_COLUMNS.items():
    normalized_dtypes.extend(
        {"table": table_name, "column": column, "dtype": str(frames[table_name][column].dtype)}
        for column in columns
    )
for table_name, column in ZIP_COLUMNS.items():
    normalized_dtypes.append(
        {"table": table_name, "column": column, "dtype": str(frames[table_name][column].dtype)}
    )
display(pd.DataFrame(normalized_dtypes).sort_values(["table", "column"]).reset_index(drop=True))

Coercion introduced new nulls:


,table,column,new_nulls
0,products,product_name_lenght,610
1,products,product_description_lenght,610
2,products,product_photos_qty,610
3,products,product_weight_g,2
4,products,product_length_cm,2
5,products,product_height_cm,2
6,products,product_width_cm,2
7,orders,order_approved_at,160
8,orders,order_delivered_carrier_date,1783
9,orders,order_delivered_customer_date,2965


,table,column,dtype
0,customers,customer_zip_code_prefix,string
1,geolocation,geolocation_lat,float64
2,geolocation,geolocation_lng,float64
3,geolocation,geolocation_zip_code_prefix,string
4,order_items,freight_value,float64
5,order_items,order_item_id,float64
6,order_items,price,float64
7,order_items,shipping_limit_date,datetime64[us]
8,orders,order_approved_at,datetime64[us]
9,orders,order_delivered_carrier_date,datetime64[us]


## 5. Define the GX checks

The raw row-count and distribution bands detect material changes from the current Olist baseline. Review and reset these bands when the source scope changes.

Observation checks use tolerances for known Olist source anomalies and report unexpected row counts.

In [57]:
E = gx.expectations

# values to verify comes from official domain values and profiling of the current Olist raw data
BRAZIL_STATES = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG",
    "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

# values for order statuses and payment types coming from distinct values in dataset
ORDER_STATUSES = [
    "created", "approved", "invoiced", "processing", "shipped", "delivered", "unavailable", "canceled",
]
# since "not_defined" percentage is very small (i.e. only 3 in dataset for now), 
# we will monitor first and add a check to ensure it's less than 0.01%
PAYMENT_TYPES = ["credit_card", "boleto", "voucher", "debit_card"]

# having the baseline helps to detect problems where only a part of CSV was loaded
RAW_ROW_BASELINES = {
    "customers": 99_441,
    "orders": 99_441,
    "order_items": 112_650,
    "payments": 103_886,
    "reviews": 99_224,
    "products": 32_951,
    "sellers": 3_095,
    "geolocation": 1_000_163,
}


def non_null(*columns):
    return [E.ExpectColumnValuesToNotBeNull(column=column) for column in columns]


def non_negative(*columns):
    return [E.ExpectColumnValuesToBeBetween(column=column, min_value=0.0) for column in columns]


# used to get data that are invalid or logically impossible
critical_expectations = {
    "customers": [
        E.ExpectColumnValuesToBeInSet(column="customer_state", value_set=BRAZIL_STATES),
        E.ExpectColumnValuesToMatchRegex(column="customer_zip_code_prefix", regex=r"^\d{5}$"),
    ],
    "orders": [
        E.ExpectColumnValuesToBeInSet(column="order_status", value_set=ORDER_STATUSES),
        E.ExpectColumnValuesToNotBeNull(column="order_purchase_timestamp"),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="order_approved_at", column_B="order_purchase_timestamp",
            or_equal=True, ignore_row_if="either_value_is_missing",
        ),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="order_delivered_customer_date", column_B="order_purchase_timestamp",
            or_equal=True, ignore_row_if="either_value_is_missing",
        ),
    ],
    "order_items": [
        E.ExpectColumnValuesToBeBetween(column="order_item_id", min_value=1),
        E.ExpectColumnValuesToBeBetween(column="price", min_value=0.0, strict_min=True),
        E.ExpectColumnValuesToBeBetween(column="freight_value", min_value=0.0),
    ],
    "payments": [
        E.ExpectColumnValuesToBeBetween(column="payment_sequential", min_value=1.0),
        E.ExpectColumnValuesToBeBetween(column="payment_value", min_value=0.0),
        # check that the number of "not_defined" payment types is less than 0.01%
        E.ExpectColumnValuesToBeInSet(column="payment_type", value_set=PAYMENT_TYPES, mostly=0.9999),
    ],
    "reviews": [
        E.ExpectColumnValuesToBeBetween(column="review_score", min_value=1.0, max_value=5.0),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="review_answer_timestamp", column_B="review_creation_date",
            or_equal=True, ignore_row_if="either_value_is_missing",
        ),
    ],
    "products": [
        *non_negative(
            "product_name_lenght", "product_description_lenght", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
        ),
    ],
    "sellers": [
        E.ExpectColumnValuesToBeInSet(column="seller_state", value_set=BRAZIL_STATES),
        E.ExpectColumnValuesToMatchRegex(column="seller_zip_code_prefix", regex=r"^\d{5}$"),
    ],
    "geolocation": [
        E.ExpectColumnValuesToBeBetween(column="geolocation_lat", min_value=-90.0, max_value=90.0),
        E.ExpectColumnValuesToBeBetween(column="geolocation_lng", min_value=-180.0, max_value=180.0),
        E.ExpectColumnValuesToBeInSet(column="geolocation_state", value_set=BRAZIL_STATES),
        E.ExpectColumnValuesToMatchRegex(column="geolocation_zip_code_prefix", regex=r"^\d{5}$"),
    ],
}

In [58]:
# monitor changes or known imperfections that 
# require investigation but do not necessarily make the data unusable
observation_expectations = {
    "customers": [
        E.ExpectColumnValuesToNotBeNull(column="customer_city", mostly=0.99),
    ],
    "orders": [
        E.ExpectColumnValuesToNotBeNull(column="order_approved_at", mostly=0.98),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="order_delivered_carrier_date", column_B="order_purchase_timestamp",
            or_equal=True, mostly=0.998, ignore_row_if="either_value_is_missing",
        ),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="order_delivered_customer_date", column_B="order_delivered_carrier_date",
            or_equal=True, mostly=0.999, ignore_row_if="either_value_is_missing",
        ),
        E.ExpectColumnPairValuesAToBeGreaterThanB(
            column_A="order_estimated_delivery_date", column_B="order_delivered_customer_date",
            or_equal=True, mostly=0.90, ignore_row_if="either_value_is_missing",
        ),
        E.ExpectColumnValuesToBeInSet(column="delivery_date_matches_status", value_set=[True], mostly=0.999),
    ],
    "order_items": [
        E.ExpectColumnMeanToBeBetween(column="price", min_value=95.0, max_value=145.0),
        E.ExpectColumnMeanToBeBetween(column="freight_value", min_value=16.0, max_value=24.0),
    ],
    "payments": [
        E.ExpectColumnValuesToBeBetween(column="payment_installments", min_value=1.0, mostly=0.9999),
        E.ExpectColumnValuesToBeBetween(column="payment_value", min_value=0.0, strict_min=True, mostly=0.9999),
        E.ExpectColumnMeanToBeBetween(column="payment_value", min_value=120.0, max_value=190.0),
    ],
    "reviews": [
        E.ExpectColumnValuesToNotBeNull(column="review_comment_title", mostly=0.10),
        E.ExpectColumnValuesToNotBeNull(column="review_comment_message", mostly=0.35),
        E.ExpectColumnMeanToBeBetween(column="review_score", min_value=3.8, max_value=4.3),
    ],
    "products": [
        E.ExpectColumnValuesToNotBeNull(column="product_category_name", mostly=0.97),
        E.ExpectColumnValuesToBeBetween(column="product_weight_g", min_value=0.0, max_value=50_000.0, mostly=0.999),
    ],
    "sellers": [
        E.ExpectColumnValuesToNotBeNull(column="seller_city", mostly=0.99),
    ],
    "geolocation": [
        E.ExpectColumnValuesToBeBetween(column="geolocation_lat", min_value=-34.0, max_value=6.0, mostly=0.9999),
        E.ExpectColumnValuesToBeBetween(column="geolocation_lng", min_value=-74.0, max_value=-34.0, mostly=0.9999),
        E.ExpectCompoundColumnsToBeUnique(
            column_list=[
                "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
                "geolocation_city", "geolocation_state",
            ],
            # TODO: monitor if value needs to be tightened
            mostly=0.60,
        ),
    ],
}

if DATA_LAYER == "raw":
    for table_name, baseline in RAW_ROW_BASELINES.items():
        observation_expectations[table_name].insert(
            0,
            E.ExpectTableRowCountToBeBetween(
                min_value=int(baseline * 0.95),
                max_value=int(baseline * 1.05),
            ),
        )

check_inventory = pd.DataFrame(
    [
        {"table": table, "critical_checks": len(critical_expectations[table]), "observation_checks": len(observation_expectations[table])}
        for table in frames
    ]
)
display(check_inventory)
print("Total checks:", check_inventory[["critical_checks", "observation_checks"]].to_numpy().sum())

,table,critical_checks,observation_checks
0,customers,2,2
1,orders,4,6
2,order_items,3,3
3,payments,3,4
4,reviews,2,4
5,products,7,3
6,sellers,2,2
7,geolocation,4,4


Total checks: 55


## 6. Register DataFrame batches, suites, validation definitions, and checkpoints

Rerunning this cell keeps the suites, validation definitions, and checkpoints in sync with the expectations defined above. Each checkpoint wraps one validation definition and runs an `UpdateDataDocsAction`, which refreshes a browsable HTML report under `gx/uncommitted/data_docs/`.

In [59]:
# creates or reuses a GX pandas data source that is created in section 3
data_source = context.data_sources.add_or_update_pandas(name="olist_dataframes")

# create 2 GX objectis for each table
def get_batch_definition(table_name):
    asset_name = f"{table_name}_dataframe_asset"
    batch_name = f"{table_name}_whole_dataframe"
    # GX 1.22 raises LookupError/KeyError here, not ValueError as some docs show -
    # confirmed against this project's installed version before relying on it.
    try:
        asset = data_source.get_asset(asset_name)
    except LookupError:
        asset = data_source.add_dataframe_asset(name=asset_name)
    try:
        return asset.get_batch_definition(batch_name)
    except KeyError:
        return asset.add_batch_definition_whole_dataframe(batch_name)

# creates or updates the expectation suite and validation definition.
# add_or_update fully replaces the suite's expectation list on every call (verified
# empirically), so rerunning this cell always reflects the current expectations above.
def replace_suite_and_validation(table_name, group, expectations, batch_definition):
    suite_name = f"{table_name}_{group}_suite"
    validation_name = f"{table_name}_{group}_validation"

    suite = gx.ExpectationSuite(name=suite_name)
    for expectation in expectations:
        expectation_copy = expectation.copy(deep=True)
        expectation_copy.id = None
        suite.add_expectation(expectation_copy)
    suite = context.suites.add_or_update(suite)

    validation = context.validation_definitions.add_or_update(
        gx.ValidationDefinition(data=batch_definition, suite=suite, name=validation_name)
    )
    return validation


# wraps a validation definition in a Checkpoint with UpdateDataDocsAction, so every run
# also refreshes the browsable HTML validation report under gx/uncommitted/data_docs/
def replace_checkpoint(table_name, group, validation):
    checkpoint_name = f"{table_name}_{group}_checkpoint"
    return context.checkpoints.add_or_update(
        gx.Checkpoint(
            name=checkpoint_name,
            validation_definitions=[validation],
            actions=[gx.checkpoint.UpdateDataDocsAction(name="update_data_docs")],
            result_format={"result_format": "SUMMARY", "partial_unexpected_count": 10},
        )
    )


batch_definitions = {table: get_batch_definition(table) for table in frames}
validations = {"critical": {}, "observation": {}}
checkpoints = {"critical": {}, "observation": {}}

for table_name in frames:
    validations["critical"][table_name] = replace_suite_and_validation(
        table_name, "critical", critical_expectations[table_name], batch_definitions[table_name]
    )
    validations["observation"][table_name] = replace_suite_and_validation(
        table_name, "observation", observation_expectations[table_name], batch_definitions[table_name]
    )
    checkpoints["critical"][table_name] = replace_checkpoint(
        table_name, "critical", validations["critical"][table_name]
    )
    checkpoints["observation"][table_name] = replace_checkpoint(
        table_name, "observation", validations["observation"][table_name]
    )

print(
    "Registered", sum(len(group) for group in validations.values()), "validation definitions and",
    sum(len(group) for group in checkpoints.values()), "checkpoints.",
)

Registered 16 validation definitions and 16 checkpoints.


## 7. Run every validation

Each validation runs through its checkpoint, which receives that table's DataFrame at runtime and refreshes the Data Docs site alongside producing the validation result.

In [60]:
validation_results = []

for group, table_checkpoints in checkpoints.items():
    for table_name, checkpoint in table_checkpoints.items():
        checkpoint_result = checkpoint.run(batch_parameters={"dataframe": frames[table_name]})
        result = next(iter(checkpoint_result.run_results.values()))
        validation_results.append(
            {
                "group": group,
                "table": table_name,
                "validation": validations[group][table_name].name,
                "success": bool(result.success),
                "result": result,
            }
        )

validation_summary = pd.DataFrame(
    [{key: item[key] for key in ["group", "table", "validation", "success"]} for item in validation_results]
).sort_values(["group", "table"]).reset_index(drop=True)

display(validation_summary)

Calculating Metrics:   0%|          | 0/20 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/27 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/16 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/15 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/16 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

,group,table,validation,success
0,critical,customers,customers_critical_validation,True
1,critical,geolocation,geolocation_critical_validation,True
2,critical,order_items,order_items_critical_validation,True
3,critical,orders,orders_critical_validation,True
4,critical,payments,payments_critical_validation,True
5,critical,products,products_critical_validation,True
6,critical,reviews,reviews_critical_validation,True
7,critical,sellers,sellers_critical_validation,True
8,observation,customers,customers_observation_validation,True
9,observation,geolocation,geolocation_observation_validation,True


## 8. Inspect failures and monitoring metrics

The detail table shows unexpected counts and percentages even when a tolerance allows the overall expectation to pass. This is where the known Olist anomalies remain visible.

In [61]:
detail_rows = []
for validation_item in validation_results:
    payload = validation_item["result"].to_json_dict()
    for expectation_result in payload["results"]:
        config = expectation_result["expectation_config"]
        result = expectation_result.get("result", {})
        detail_rows.append(
            {
                "group": validation_item["group"],
                "table": validation_item["table"],
                "expectation": config.get("type"),
                "column": config.get("kwargs", {}).get("column", "<table or column pair>"),
                "success": expectation_result.get("success"),
                "observed_value": result.get("observed_value"),
                "unexpected_count": result.get("unexpected_count"),
                "unexpected_percent": result.get("unexpected_percent"),
            }
        )

gx_details = pd.DataFrame(detail_rows)
failed_details = gx_details[gx_details["success"] == False]

print("Failed expectations:")
display(failed_details if not failed_details.empty else pd.DataFrame({"result": ["None"]}))

print("Observation metrics with unexpected rows:")
display(
    gx_details[
        (gx_details["group"] == "observation")
        & (gx_details["unexpected_count"].fillna(0) > 0)
    ].reset_index(drop=True)
)

Failed expectations:


,result
0,None


Observation metrics with unexpected rows:


,group,table,expectation,column,success,observed_value,unexpected_count,unexpected_percent
0,observation,orders,expect_column_pair_values_a_to_be_greater_than_b,<table or column pair>,True,NaN,166.0,0.169981
1,observation,orders,expect_column_pair_values_a_to_be_greater_than_b,<table or column pair>,True,NaN,23.0,0.023840
2,observation,orders,expect_column_pair_values_a_to_be_greater_than_b,<table or column pair>,True,NaN,7827.0,8.112899
3,observation,orders,expect_column_values_to_not_be_null,order_approved_at,True,NaN,160.0,0.160899
4,observation,orders,expect_column_values_to_be_in_set,delivery_date_matches_status,True,NaN,14.0,0.014079
5,observation,payments,expect_column_values_to_be_between,payment_installments,True,NaN,2.0,0.001925
6,observation,payments,expect_column_values_to_be_between,payment_value,True,NaN,9.0,0.008663
7,observation,geolocation,expect_compound_columns_to_be_unique,<table or column pair>,True,NaN,390005.0,38.994144
8,observation,geolocation,expect_column_values_to_be_between,geolocation_lat,True,NaN,31.0,0.003099
9,observation,geolocation,expect_column_values_to_be_between,geolocation_lng,True,NaN,37.0,0.003699


## 9. Apply the pipeline gate

Only critical validation failures should stop ingestion. Observation failures should create a warning or alert for review. The notebook leaves `RAISE_ON_CRITICAL_FAILURE` off so you can inspect all results during development.

In [62]:
critical_failures = validation_summary[
    (validation_summary["group"] == "critical") & (~validation_summary["success"])
]
observation_failures = validation_summary[
    (validation_summary["group"] == "observation") & (~validation_summary["success"])
]

print(f"Critical validation failures: {len(critical_failures)}")
print(f"Observation validation failures: {len(observation_failures)}")

# can change the settings in .env file for future orchestrator (i.e. dagster to turn on or off)
RAISE_ON_CRITICAL_FAILURE = os.getenv("GX_RAISE_ON_CRITICAL_FAILURE", "false").lower() == "true"
if RAISE_ON_CRITICAL_FAILURE and not critical_failures.empty:
    raise RuntimeError("Critical GX checks failed. Review critical_failures and failed_details.")

Critical validation failures: 0
Observation validation failures: 0


## 10. What moves into Dagster later

Dagster should call the same sequence during each pipeline run:

1. Load the current tables from BigQuery.
2. Normalize the validation dtypes.
3. Run each validation definition with that table's DataFrame.
4. Raise an error only when `critical_failures` is not empty.
5. Store or alert on observation results so changes can be compared across runs.

Move the reusable validation code into a Python module under `gx/` when the orchestration assets are implemented.

Category translation and holiday checks will be implemented in dbt once the staging models exist.